In [30]:
import pandas as pd
df = pd.read_csv("retrievable_peptides.tsv", sep="\t")
df.head()

,precursor_protein,evidence_peptides_scans
0,A0A087WXM9,"[('CASNSESDNAACEILLAEK', 253, 'non_standard_mo..."
1,O15172,"[('TVISEEGIGCF', 1763, 'non_standard_modificat..."
2,A6NNF4,"[('CLDTAQKNLY', 1027, 'non_standard_modificati..."
3,P25021,"[('APNGTASSFCLDSTACK', 5159, 'non_standard_mod..."
4,P0C263,"[('CAAQLASALEYIHAR', 3643, 'non_standard_modif..."


In [31]:
df_pa = pd.read_csv("all_usi_Nov2025.xlsx - all_usi.tsv", sep="\t", usecols=["Dataset", "DemodPeptide"])
df_pa.head()

,Dataset,DemodPeptide
0,PXD006633,EIVMTQSPDTLSVSPGER
1,PXD006633,EIVMTQSPDTLSVSPGER
2,PXD006633,EIVMTQSPDTLSVSPGER
3,PXD006633,EIVMTQSPDTLSVSPGER
4,PXD006633,EIVMTQSPDTLSVSPGER


In [32]:
import ast
def _parse_evidence(peptide_entry):
    if peptide_entry is None:
        return []
    text = str(peptide_entry).strip()
    if not text:
        return []
    start = text.find('[')
    if start >= 0:
        normalized = text[start:]
    else:
        return []
    normalized = normalized.replace("nan", "None")
    if not normalized.endswith(']'):
        normalized = normalized + ']'
    try:
        return ast.literal_eval(normalized)
    except (ValueError, SyntaxError):
        return []

output a dataset_level.tsv that look like this, with the peptide dataset matching above: Dataset	num_retrieved_protein	specific_proteins
PXD019643	14	A0A0B4J271;A2RRH5;A4D0T7;A5LHX3;A6NNF4;O00270;O15172;Q3ZCN5;Q96KX1;Q99616;Q9BQJ4;Q9BTD3;Q9GZV3;Q9NQ39
PXD008333	10	A6NHN6;A8MPX8;B3SHH9;P0DTF9;Q1W4C9;Q30KQ8;Q6NXP0;Q8NG35;Q96J77;Q9NP94
PXD013649	7	A2RRH5;A6NNF4;O15172;Q6MZN7;Q6ZW05;Q8N6I4;Q9BRJ9
PXD004894	6	A6NNF4;Q6MZN7;Q6UX40;Q8N2M4;Q9BSJ1;Q9BTD3
PXD010154	6	A0A5B6;O95626;Q5VYV0;Q6UX40;Q6ZUT3;Q96DS6
PXD020079	6	A2RRH5;A4D0T7;A6NNF4;O15172;Q96N22;Q9BQJ4
PXD022150	6	A2RRH5;A4D0T7;A6NNF4;O15172;Q6UX40;Q9BTD3, but what i also want is that rank the num retrive protein from high to low, and if its in high, remove it in lower sets to form a union set of proteins

In [33]:
# Merge evidence peptides with dataset peptides
merged = evidence_df.merge(
    df_pa,
    left_on="peptide",
    right_on="DemodPeptide",
    how="inner"
)

# Build initial per-dataset protein sets
dataset_proteins = (
    merged.groupby("Dataset")["precursor_protein"]
    .apply(lambda s: set(s.dropna()))
)

# Rank datasets by number of retrieved proteins (desc)
ranked = dataset_proteins.apply(len).sort_values(ascending=False)

# Build union-ranked sets: remove proteins already assigned to higher-ranked datasets
seen = set()
rows = []
for ds in ranked.index:
    proteins = dataset_proteins[ds] - seen
    seen |= proteins
    rows.append(
        {
            "Dataset": ds,
            "num_retrieved_protein": len(proteins),
            "specific_proteins": ";".join(sorted(proteins))
        }
    )

result = pd.DataFrame(rows)

# Save to TSV
result.to_csv("dataset_level.tsv", sep="\t", index=False)
result.head()

,Dataset,num_retrieved_protein,specific_proteins
0,PXD019643,27,A0A0B4J271;A2RRH5;A4D0T7;A5LHX3;A6NNF4;O00270;...
1,PXD013649,12,O75445;Q01726;Q6MZN7;Q6ZMC9;Q6ZW05;Q8N6I4;Q8TD...
2,PXD006939,9,A0A0K0K1B3;A0A584;B4DXR9;O15342;O94777;O95944;...
3,PXD008333,11,A6NHN6;A6NNT2;A8MPX8;B3SHH9;P0DTF9;Q1W4C9;Q30K...
4,PXD025716,5,B7Z8K6;O15178;Q14330;Q6PF15;Q86V71
